# 04 - Observabilidade
> Logs estruturados, traces e metricas

**Modulo:** `08_arquitetura` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import time, uuid, json
from dataclasses import dataclass, field
from typing import List

@dataclass
class Span:
    id:str=field(default_factory=lambda:str(uuid.uuid4())[:8])
    nome:str=''; modelo:str=''; t0:float=field(default_factory=time.time)
    tk_in:int=0; tk_out:int=0; custo:float=0.0
    @property
    def dur(self): return round(time.time()-self.t0,3)

class Trace:
    def __init__(self): self.id=f'tr-{str(uuid.uuid4())[:6]}'; self.spans:List[Span]=[]; self.t0=time.time()
    def add(self,s): self.spans.append(s)
    def resumo(self):
        return {'id':self.id,'dur':round(time.time()-self.t0,3),
                'custo':round(sum(s.custo for s in self.spans),6),
                'spans':[{'nome':s.nome,'dur':s.dur}for s in self.spans]}

PRECOS={HAIKU:(0.00000025,0.00000125),SONNET:(0.000003,0.000015)}

def ask_obs(prompt,model=HAIKU,trace=None):
    s=Span(nome='llm',modelo=model)
    r=client.messages.create(model=model,max_tokens=256,messages=[{'role':'user','content':prompt}])
    s.tk_in=r.usage.input_tokens; s.tk_out=r.usage.output_tokens
    pi,po=PRECOS.get(model,(0,0))
    s.custo=s.tk_in*pi+s.tk_out*po
    if trace: trace.add(s)
    return r.content[0].text

trace=Trace()
ask_obs('O que e Python?',trace=trace)
ask_obs('O que e ML?',trace=trace)
print(json.dumps(trace.resumo(),indent=2))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
